# Task 4 - Named Entity Recognition (NER) from News Articles
Walkthrough of the `ner_news` package: **rule-based**, **model-based (spaCy)** and **hybrid** NER on the CoNLL-2003 label scheme.

Run from the repository root after `pip install -e .` and `python -m spacy download en_core_web_sm en_core_web_md`.

In [ ]:
import pandas as pd
from spacy import displacy

from ner_news.config import SAMPLE_ARTICLES, SAMPLE_CONLL
from ner_news.data import dataset_statistics, load_dataset
from ner_news.evaluate import run_evaluation, summary_table
from ner_news.extract import extract_entities, read_articles
from ner_news.models import build_recognizers

pd.set_option("display.width", 140)

## 1. Data
Swap `SAMPLE_CONLL` for `data/raw/conll2003/test.txt` to use the real CoNLL-2003 test set (see `docs/DATASET.md`).

In [ ]:
DATA = SAMPLE_CONLL          # <- change to "data/raw/conll2003/test.txt"
sentences = load_dataset(DATA)
print(len(sentences), "sentences")
dataset_statistics(sentences)

## 2. Build the recognizers

In [ ]:
recognizers = build_recognizers(["rule", "model", "hybrid"], ["en_core_web_sm"])
[r.name for r in recognizers]

## 3. Try them on raw text

In [ ]:
text = "Dr. Ayesha Khan of the University of Karachi told Reuters that Microsoft will open an office in Islamabad."
rule, model, hybrid = recognizers
for rec in recognizers:
    doc = rec.analyze(text)
    print(f"{rec.name:28s}", [(e.text, e.label_) for e in doc.ents])

In [ ]:
displacy.render(hybrid.analyze(text), style="ent", jupyter=True)

## 4. Evaluate (entity-level, exact match)

In [ ]:
results = run_evaluation(sentences, recognizers, "outputs/notebook_run", dataset_name=str(DATA), make_plots=True)
summary_table(results)

In [ ]:
results[1].report.round(3)   # per-label report for the model-based approach

## 5. Error analysis
Where does the statistical model go wrong? (`type_error` = right span, wrong label; `boundary_error` = overlapping but different span.)

In [ ]:
model_result = results[1]
print(model_result.error_summary.to_string(index=False))
model_result.error_details.query("kind != 'correct'").head(15)[["kind", "gold", "gold_label", "pred", "pred_label"]]

## 6. Extract entities from news articles into a DataFrame

In [ ]:
articles = read_articles(SAMPLE_ARTICLES)
df = extract_entities(hybrid, articles)
df.head(15)

In [ ]:
df.groupby("label")["entity"].agg(["count", "nunique"])